In [0]:
%MD Resilient MERGE (Idempotent Loading)


In [0]:
from delta.tables import DeltaTable

def upsert_to_delta(microBatchDF, batchId):
    # Target Delta table
    target_table = DeltaTable.forPath(spark, "/mnt/delta/gold_table")
    
    # Use MERGE to handle both updates and inserts (SCD Type 1)
    target_table.alias("target") \
        .merge(
            microBatchDF.alias("source"),
            "target.id = source.id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

# Streaming entry point
streaming_df.writeStream \
    .foreachBatch(upsert_to_delta) \
    .option("checkpointLocation", "/mnt/delta/checkpoints/gold_table") \
    .start()

In [0]:
from delta.tables import DeltaTable

# 1. Define Widgets at the top of the notebook
dbutils.widgets.text("table_name", "schema.target_table", "Target Table Name")
dbutils.widgets.text("recovery_version", "", "Version to Restore (Optional)")
dbutils.widgets.text("recovery_timestamp", "", "Timestamp to Restore (Optional)")

# 2. Extract values
table_name = dbutils.widgets.get("table_name")
v_to_restore = dbutils.widgets.get("recovery_version")
ts_to_restore = dbutils.widgets.get("recovery_timestamp")

# 3. Execution Logic
delta_table = DeltaTable.forName(spark, table_name)

if v_to_restore:
    print(f"Restoring {table_name} to version: {v_to_restore}")
    delta_table.restoreToVersion(int(v_to_restore)) #
elif ts_to_restore:
    print(f"Restoring {table_name} to timestamp: {ts_to_restore}")
    delta_table.restoreToTimestamp(ts_to_restore) #
else:
    raise ValueError("You must provide either a version or a timestamp for recovery.")
